## TC 5033
## Deep Learning
## Transformers

#### Activity 4: Implementing a Translator

### Team # 73

* A00959089 - Marcelo Ismael López Verdugo
* A01796847 - Pedro Cuauhtemoc Marquez Correo
* A01625357 - Edson Alejandro Lozano García
* A01796999 - Christian Gustavo Martínez Ramírez

- Objective

To understand the Transformer Architecture by Implementing a translator.

- Instructions

    This activity requires submission in teams. While teamwork is encouraged, each member is expected to contribute individually to the assignment. The final submission should feature the best arguments and solutions from each team member. Only one person per team needs to submit the completed work, but it is imperative that the names of all team members are listed in a Markdown cell at the very beginning of the notebook (either the first or second cell). Failure to include all team member names will result in the grade being awarded solely to the individual who submitted the assignment, with zero points given to other team members (no exceptions will be made to this rule).

    Follow the provided code. The code already implements a transformer from scratch as explained in one of [week's 9 videos](https://youtu.be/XefFj4rLHgU)

    Since the provided code already implements a simple translator, your job for this assignment is to understand it fully, and document it using pictures, figures, and markdown cells.  You should test your translator with at least 10 sentences. The dataset used for this task was obtained from [Tatoeba, a large dataset of sentences and translations](https://tatoeba.org/en/downloads).
  
- Evaluation Criteria

    - Code Readability and Comments
    - Traning a translator
    - Translating at least 10 sentences.

- Submission

Submit this Jupyter Notebook in canvas with your complete solution, ensuring your code is well-commented and includes Markdown cells that explain your design choices, results, and any challenges you encountered.



#### Script to convert csv to text file 

In [1]:
#This script requires to convert the TSV file to CSV
# easiest way is to open it in Calc or excel and save as csv
#PATH = '/media/pepe/DataUbuntu/Databases/spanish_english/eng-spa2024.csv'
import pandas as pd
PATH = 'English-Spanish.tsv'
# Read TSV file
df = pd.read_table(PATH, sep='\t',on_bad_lines='skip')
# Save as CSV without index column
df.to_csv('English-Spanish.csv', index=False)
print("TSV successfully converted to CSV!")
df = pd.read_csv('English-Spanish.csv')

TSV successfully converted to CSV!


In [2]:
eng_spa_cols = df.iloc[:, [1, 3]]
eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()  
eng_spa_cols = eng_spa_cols.sort_values(by='length')  
eng_spa_cols = eng_spa_cols.drop(columns=['length'])  

output_file_path = 'eng-spa4.txt'
eng_spa_cols.to_csv(output_file_path, sep='\t', index=False, header=False)

C:\Users\MLOPE243\AppData\Local\Temp\ipykernel_480\1433477401.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eng_spa_cols['length'] = eng_spa_cols.iloc[:, 0].str.len()


## Transformer - Attention is all you need

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np
import re

torch.manual_seed(23)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [5]:
MAX_SEQ_LEN = 40 #128

In [6]:
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len = MAX_SEQ_LEN):
        super().__init__()
        self.pos_embed_matrix = torch.zeros(max_seq_len, d_model, device=device)
        token_pos = torch.arange(0, max_seq_len, dtype = torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() 
                             * (-math.log(10000.0)/d_model))
        self.pos_embed_matrix[:, 0::2] = torch.sin(token_pos * div_term)
        self.pos_embed_matrix[:, 1::2] = torch.cos(token_pos * div_term)
        self.pos_embed_matrix = self.pos_embed_matrix.unsqueeze(0).transpose(0,1)
        
    def forward(self, x):
#         print(self.pos_embed_matrix.shape)
#         print(x.shape)
        return x + self.pos_embed_matrix[:x.size(0), :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model = 512, num_heads = 8):
        super().__init__()
        assert d_model % num_heads == 0, 'Embedding size not compatible with num heads'
        
        self.d_v = d_model // num_heads
        self.d_k = self.d_v
        self.num_heads = num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, Q, K, V, mask = None):
        batch_size = Q.size(0)
        '''
        Q, K, V -> [batch_size, seq_len, num_heads*d_k]
        after transpose Q, K, V -> [batch_size, num_heads, seq_len, d_k]
        '''
        Q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        K = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        V = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2 )
        
        weighted_values, attention = self.scale_dot_product(Q, K, V, mask)
        weighted_values = weighted_values.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads*self.d_k)
        weighted_values = self.W_o(weighted_values)
        
        return weighted_values, attention
        
        
    def scale_dot_product(self, Q, K, V, mask = None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attention = F.softmax(scores, dim = -1)
        weighted_values = torch.matmul(attention, V)
        
        return weighted_values, attention
        

class PositionFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))
    
class EncoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.droupout1 = nn.Dropout(dropout)
        self.droupout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask = None):
        attention_score, _ = self.self_attn(x, x, x, mask)
        x = x + self.droupout1(attention_score)
        x = self.norm1(x)
        x = x + self.droupout2(self.ffn(x))
        return self.norm2(x)

class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderSubLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, target_mask=None, encoder_mask=None):
        attention_score, _ = self.self_attn(x, x, x, target_mask)
        x = x + self.dropout1(attention_score)
        x = self.norm1(x)
        
        encoder_attn, _ = self.cross_attn(x, encoder_output, encoder_output, encoder_mask)
        x = x + self.dropout2(encoder_attn)
        x = self.norm2(x)
        
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        return self.norm3(x)
        
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderSubLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x, encoder_output, target_mask, encoder_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, target_mask, encoder_mask)
        return self.norm(x)

In [7]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers,
                 input_vocab_size, target_vocab_size, 
                 max_len=MAX_SEQ_LEN, dropout=0.1):
        super().__init__()
        self.encoder_embedding = nn.Embedding(input_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.pos_embedding = PositionalEmbedding(d_model, max_len)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_layer = nn.Linear(d_model, target_vocab_size)
        
    def forward(self, source, target):
        # Encoder mask
        source_mask, target_mask = self.mask(source, target)
        # Embedding and positional Encoding
        source = self.encoder_embedding(source) * math.sqrt(self.encoder_embedding.embedding_dim)
        source = self.pos_embedding(source)
        # Encoder
        encoder_output = self.encoder(source, source_mask)
        
        # Decoder embedding and postional encoding
        target = self.decoder_embedding(target) * math.sqrt(self.decoder_embedding.embedding_dim)
        target = self.pos_embedding(target)
        # Decoder
        output = self.decoder(target, encoder_output, target_mask, source_mask)
        
        return self.output_layer(output)
        
        
    
    def mask(self, source, target):
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)
        size = target.size(1)
        no_mask = torch.tril(torch.ones((1, size, size), device=device)).bool()
        target_mask = target_mask & no_mask
        return source_mask, target_mask
        

#### Simple test

In [8]:
seq_len_source = 10
seq_len_target = 10
batch_size = 2
input_vocab_size = 50
target_vocab_size = 50

source = torch.randint(1, input_vocab_size, (batch_size, seq_len_source))
target = torch.randint(1, target_vocab_size, (batch_size, seq_len_target))

In [9]:
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

model = Transformer(d_model, num_heads, d_ff, num_layers,
                  input_vocab_size, target_vocab_size, 
                  max_len=MAX_SEQ_LEN, dropout=0.1)

model = model.to(device)
source = source.to(device)
target = target.to(device)

In [10]:
output = model(source, target)

In [11]:
# Expected output shape -> [batch, seq_len_target, target_vocab_size] i.e. [2, 10, 50]
print(f'ouput.shape {output.shape}')

ouput.shape torch.Size([2, 10, 50])


### Translator Eng-Spa

In [12]:
PATH = 'eng-spa4.txt'

In [13]:
with open(PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()
eng_spa_pairs = [line.strip().split('\t') for line in lines if '\t' in line]

In [14]:
eng_spa_pairs[:10]

[['Go.', 'Vayan.'],
 ['Go!', '¡Sal!'],
 ['Go!', '¡Váyase!'],
 ['Go!', '¡Ya!'],
 ['Go.', 'Vaya.'],
 ['Go.', 'Vete.'],
 ['Go.', 'Váyanse.'],
 ['So?', '¿Entonces?'],
 ['Go.', 'Ve.'],
 ['Go!', '¡Ve!']]

In [15]:
eng_sentences = [pair[0] for pair in eng_spa_pairs]
spa_sentences = [pair[1] for pair in eng_spa_pairs]

In [16]:
print(eng_sentences[:10])
print(spa_sentences[:10])


['Go.', 'Go!', 'Go!', 'Go!', 'Go.', 'Go.', 'Go.', 'So?', 'Go.', 'Go!']
['Vayan.', '¡Sal!', '¡Váyase!', '¡Ya!', 'Vaya.', 'Vete.', 'Váyanse.', '¿Entonces?', 'Ve.', '¡Ve!']


In [17]:
def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[á]+", "a", sentence)
    sentence = re.sub(r"[é]+", "e", sentence)
    sentence = re.sub(r"[í]+", "i", sentence)
    sentence = re.sub(r"[ó]+", "o", sentence)
    sentence = re.sub(r"[ú]+", "u", sentence)
    sentence = re.sub(r"[^a-z]+", " ", sentence)
    sentence = sentence.strip()
    sentence = '<sos> ' + sentence + ' <eos>'
    return sentence

In [18]:
s1 = '¿Hola @ cómo estás? 123'

In [19]:
print(s1)
print(preprocess_sentence(s1))

¿Hola @ cómo estás? 123
<sos> hola como estas <eos>


In [20]:
eng_sentences = [preprocess_sentence(sentence) for sentence in eng_sentences]
spa_sentences = [preprocess_sentence(sentence) for sentence in spa_sentences]

In [21]:
spa_sentences[:10]

['<sos> vayan <eos>',
 '<sos> sal <eos>',
 '<sos> vayase <eos>',
 '<sos> ya <eos>',
 '<sos> vaya <eos>',
 '<sos> vete <eos>',
 '<sos> vayanse <eos>',
 '<sos> entonces <eos>',
 '<sos> ve <eos>',
 '<sos> ve <eos>']

In [22]:
#def build_vocab(sentences,max_vocab=800):
#    words = [word for sentence in sentences for word in sentence.split()]
#    word_count = Counter(words)
#    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)
#    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_counts[:max_vocab], 2)}
#    word2idx['<pad>'] = 0
#    word2idx['<unk>'] = 1
#    idx2word = {idx: word for word, idx in word2idx.items()}
#    return word2idx, idx2word

In [23]:
def build_vocab(sentences, max_vocab=8000):
    words = [word for sentence in sentences for word in sentence.split()]
    word_count = Counter(words)
    sorted_word_counts = sorted(word_count.items(), key=lambda x:x[1], reverse=True)[:max_vocab]

    word2idx = {word: idx+2 for idx, (word, _) in enumerate(sorted_word_counts)}
    word2idx['<pad>'] = 0
    word2idx['<unk>'] = 1

    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word

In [24]:
#Test data with reduced sentences
MAX_DATASET = 10000 
eng_sent_test=eng_sentences[:MAX_DATASET]
spa_sent_test=spa_sentences[:MAX_DATASET]
eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
spa_word2idx, spa_idx2word = build_vocab(spa_sentences)
eng_vocab_size = len(eng_word2idx)
spa_vocab_size = len(spa_word2idx)

In [25]:
print(eng_vocab_size, spa_vocab_size)

8002 8002


In [26]:
class EngSpaDataset(Dataset):
    def __init__(self, eng_sentences, spa_sentences, eng_word2idx, spa_word2idx):
        self.eng_sentences = eng_sentences
        self.spa_sentences = spa_sentences
        self.eng_word2idx = eng_word2idx
        self.spa_word2idx = spa_word2idx
        
    def __len__(self):
        return len(self.eng_sentences)
    
    def __getitem__(self, idx):
        eng_sentence = self.eng_sentences[idx]
        spa_sentence = self.spa_sentences[idx]
        # return tokens idxs
        eng_idxs = [self.eng_word2idx.get(word, self.eng_word2idx['<unk>']) for word in eng_sentence.split()]
        spa_idxs = [self.spa_word2idx.get(word, self.spa_word2idx['<unk>']) for word in spa_sentence.split()]
        
        return torch.tensor(eng_idxs), torch.tensor(spa_idxs)

In [27]:
def collate_fn(batch):
    eng_batch, spa_batch = zip(*batch)
    eng_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in eng_batch]
    spa_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spa_batch]
    eng_batch = torch.nn.utils.rnn.pad_sequence(eng_batch, batch_first=True, padding_value=0)
    spa_batch = torch.nn.utils.rnn.pad_sequence(spa_batch, batch_first=True, padding_value=0)
    return eng_batch, spa_batch
    

In [28]:
def train(model, dataloader, loss_function, optimiser, epochs):
    model.train()
    for epoch in range(epochs):
        total_loss = 0 
        for i, (eng_batch, spa_batch) in enumerate(dataloader):
            eng_batch = eng_batch.to(device)
            spa_batch = spa_batch.to(device)
            # Decoder preprocessing
            target_input = spa_batch[:, :-1]
            target_output = spa_batch[:, 1:].contiguous().view(-1)
            # Zero grads
            optimiser.zero_grad()
            # run model
            output = model(eng_batch, target_input)
            output = output.view(-1, output.size(-1))
            # loss\
            loss = loss_function(output, target_output)
            # gradient and update parameters
            loss.backward()
            optimiser.step()
            total_loss += loss.item()
            
            #visible progress
            if i % 100 == 0:
                print(f"Epoch {epoch} | Batch {i}/{len(dataloader)} | Loss: {loss.item():.4f}")
        avg_loss = total_loss/len(dataloader)
        print(f'Epoch: {epoch}/{epochs}, Loss: {avg_loss:.4f}')
            
            

In [29]:
BATCH_SIZE = 32 #64
dataset = EngSpaDataset(eng_sentences, spa_sentences, eng_word2idx, spa_word2idx)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
#dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,num_workers=2,pin_memory=False)

In [30]:
d_model = 128 #512
num_heads = 4 #8
d_ff = 512 #2048
num_layers = 2 #6
model = Transformer(d_model, num_heads, d_ff, num_layers,
                    input_vocab_size=eng_vocab_size, target_vocab_size=spa_vocab_size,
                    max_len=MAX_SEQ_LEN, dropout=0.1)

In [31]:
model = model.to(device)
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimiser = optim.Adam(model.parameters(), lr=0.0001)


In [32]:
train(model, dataloader, loss_function, optimiser, epochs = 10)

Epoch 0 | Batch 0/8752 | Loss: 9.1469
Epoch 0 | Batch 100/8752 | Loss: 6.9880
Epoch 0 | Batch 200/8752 | Loss: 6.3702
Epoch 0 | Batch 300/8752 | Loss: 5.6297
Epoch 0 | Batch 400/8752 | Loss: 5.5154
Epoch 0 | Batch 500/8752 | Loss: 5.7007
Epoch 0 | Batch 600/8752 | Loss: 5.3255
Epoch 0 | Batch 700/8752 | Loss: 5.4290
Epoch 0 | Batch 800/8752 | Loss: 5.3287
Epoch 0 | Batch 900/8752 | Loss: 5.1382
Epoch 0 | Batch 1000/8752 | Loss: 5.1148
Epoch 0 | Batch 1100/8752 | Loss: 5.1478
Epoch 0 | Batch 1200/8752 | Loss: 4.9872
Epoch 0 | Batch 1300/8752 | Loss: 5.0195
Epoch 0 | Batch 1400/8752 | Loss: 5.0697
Epoch 0 | Batch 1500/8752 | Loss: 5.1285
Epoch 0 | Batch 1600/8752 | Loss: 4.9623
Epoch 0 | Batch 1700/8752 | Loss: 5.0084
Epoch 0 | Batch 1800/8752 | Loss: 4.6952
Epoch 0 | Batch 1900/8752 | Loss: 4.6653
Epoch 0 | Batch 2000/8752 | Loss: 4.5690
Epoch 0 | Batch 2100/8752 | Loss: 4.6839
Epoch 0 | Batch 2200/8752 | Loss: 4.6884
Epoch 0 | Batch 2300/8752 | Loss: 4.3952
Epoch 0 | Batch 2400/8752 | 

3 epochs  52 m 29 seconds --> Using only 200 in test size, incresing to 5k for second test   
3 epochs  55 m 27 seconds --> Traduction successful :   
Epoch: 0/3, Loss: 2.8605  
Epoch: 1/3, Loss: 2.1734  
Epoch: 2/3, Loss: 1.9703  
3 epochs  90 m 18 seconds --> No limit in the test size  
Epoch: 0/3, Loss: 4.3518  
Epoch: 1/3, Loss: 3.1737  
Epoch: 2/3, Loss: 2.6766  
10 epochs 274 m 0 seconds -->  Max sequence lenght to 40 (128 original), dataloader extra configs   
Epoch: 9/10, Loss: 1.7354  

In [33]:
def sentence_to_indices(sentence, word2idx):
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)

In [34]:
def evaluate_translations(model, sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    "Hello, how are you?",
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "Good night!",
    "My name is John."
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Evaluate translations
evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)


Input sentence: Hello, how are you?
Traducción: <sos> hola como estas <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo la inteligencia artificial <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> la inteligencia artificial es genial <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

Input sentence: My name is John.
Traducción: <sos> john es mi nombre <eos>



#### First trial 
Input sentence: Hello, how are you?
Traducción: <sos> como <unk> <unk> <unk> <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> <unk> <unk> <unk> <unk> <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> <unk> <unk> <unk> <unk> <eos>

Input sentence: Good night!
Traducción: <sos> <unk> <unk> <eos>

#### Second trial
5k dictionary, 3 epochs
Input sentence: Hello, how are you?
Traducción: <sos> como estas <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> soy <unk> <unk> <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> el <unk> es muy grande <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

#### Third trial
Full dictionary, 3 epochs   
Input sentence: Hello, how are you?
Traducción: <sos> como estas <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo <unk> <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> la inteligencia es un gran gran gran gran gran gran gran gran gran <unk> <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

#### Fourth trial
Input sentence: Hello, how are you?
Traducción: <sos> hola como estas <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo la inteligencia artificial <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> la inteligencia artificial es genial <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

Input sentence: My name is John.
Traducción: <sos> john es mi nombre <eos>

# Saving the model

In [35]:
from datetime import datetime
date=datetime.now().strftime("%Y%m%d_%H%M%S")
name="translator_model"+date
torch.save(model.state_dict(), f"{name}.pth")

# Using a previously saved model

## Loading a model

In [ ]:
model = Transformer(
    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=2,
    input_vocab_size=eng_vocab_size,
    target_vocab_size=spa_vocab_size,
    max_len=MAX_SEQ_LEN,
    dropout=0.1
)

model.load_state_dict(torch.load("translator_model20260323_044641.pth", map_location=device))
model.to(device)
model.eval()


Transformer(
  (encoder_embedding): Embedding(8002, 128)
  (decoder_embedding): Embedding(8002, 128)
  (pos_embedding): PositionalEmbedding()
  (encoder): Encoder(
    (layers): ModuleList(
      (0-1): 2 x EncoderSubLayer(
        (self_attn): MultiHeadAttention(
          (W_q): Linear(in_features=128, out_features=128, bias=True)
          (W_k): Linear(in_features=128, out_features=128, bias=True)
          (W_v): Linear(in_features=128, out_features=128, bias=True)
          (W_o): Linear(in_features=128, out_features=128, bias=True)
        )
        (ffn): PositionFeedForward(
          (linear1): Linear(in_features=128, out_features=512, bias=True)
          (linear2): Linear(in_features=512, out_features=128, bias=True)
        )
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (droupout1): Dropout(p=0.1, inplace=False)
        (droupout2): Dropout(p=0.1, inplace=False)
      )

## Testing loaded model

In [ ]:
# Example sentences to test the translator
test_sentences = [
    "Hello, how are you?",
    "I am learning artificial intelligence.",
    "Artificial intelligence is great.",
    "Good night!",
    "My name is John.",
    "I am doing my best"
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Evaluate translations
evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)

Input sentence: Hello, how are you?
Traducción: <sos> hola como estas <eos>

Input sentence: I am learning artificial intelligence.
Traducción: <sos> estoy aprendiendo la inteligencia artificial <eos>

Input sentence: Artificial intelligence is great.
Traducción: <sos> la inteligencia artificial es genial <eos>

Input sentence: Good night!
Traducción: <sos> buenas noches <eos>

Input sentence: My name is John.
Traducción: <sos> john es mi nombre <eos>

Input sentence: I am doing my best
Traducción: <sos> estoy haciendo mi mejor esfuerzo <eos>

